<a href="https://colab.research.google.com/github/zhangling297/deep-learning-with-python-notebooks/blob/master/Fully_connected_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

fully- connected NN for binary classification - using heart disease data - upload the data, subset the columns to the 3 objects that we are interested in, and then standardize the data.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler


df_heart_disease = pd.read_csv('/heart_disease.csv')

print(df_heart_disease)

In [ ]:

# Define the predictor variabies and target variable form the subset
X = df_heart_disease[['chol', 'age', 'trestbps']]
y = df_heart_disease['target']

#Initialize the StandardScaler
scaler = StandardScaler()

#Fit the scaler to the data and transform it

X_scaled = scaler.fit_transform(X)

# Convert the scaled array back to a DataFrame  for easier viewing
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

#check what it looks like
display(X_scaled.head())

# Now we want to define the model. The following code block is blank for you to define a model in Torch: Attempt to write a class for a fully-connected NN with one hidden layer with 8 neurons.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class FullyConnectedNN(nn.Module):
  def_init_(self, input_size, hidden_size=8):
    super(FullyConnectedNN, self)._init_()
    self.fc1 = nn.Linear(input_size, hidden_size)
    self.fc2 = nn.Linear(hidden_size, 1)
    self.sigmoid = nn.Sigmoid()
    self.relu = nn.ReLu()

  def forward(self, x):
    z1 = self.fc1(x)
    a1 = self.relu(z1)
    z2 = self.fc2(a1)
    return self.sigmoid(z2)


# Now create the training loop where you learn the parameters wiht gradient descent

In [ ]:
# To use PyTorch you mist convert our data to PyTorch tensors
X_torch = torch.tensor(X_scaled.values,dtype=torch.float32)
y_torch=torch.tensor(y.values, dtype=torch.float32).view(-1, 1)

#Initialize the model we defined above
model = FullyConnectedNN(input_size=3)

#Print the random initial values of the weights and bias
with torch.no_grad():
  print('Initial Values: Weights = {model.fc1.weight.numpy()[0].round(decimals=3)}, \
  Bias = {model.fc1.bias.numpy().round(decimals=3)}')
# Define our Loss function and algorithm for optimization (aka our 'Optimizer')
# BCELoss = Binary Cross Entropy Loss(standard for binary classification)
criterion = nn.BCELoss()

# This is the function for "Stocastic' Grandient Descent not plain Gradient Descent. Below does not add any of the Stochastic elements(randominess) so it will act exactly as what appear here
optimizer = optim.SGD(model.parameters(), lr=0.05)

# The following is the training loop, the number of"epochs"is is the number of times we want our whole dataset to propagate forward through the model. Here the number of epocs is also the number of steps of gradient descent will be made

epochs = 10_000

print()
print('Starting Training with Gradient Descent')
print()
for epoch in range(epochs):

  #pass the entire dataset X at once to the model and calcualte predictions
  y_pred = model(X_torch)

  # note here we actually calculate the cost not the loss but all PyTorch syntax calls this the loss
  loss = criterion(y_pred, y_torch)

  # These steps (1) delete any previous gradient calculations; 2) do backpropagation and 3) update the model parameters
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if (epoch +1) % 50 == 0:
    # the below line prevents Torch from remembering the following,
    # slower with larger datasets
    with torch.no_grad():
      predicted_labels = (y_pred > 0.5).float()
      accuracy = (predicted_labels == y_torch).float().mean()
      print(f'Epoch {epoch+1}: Loss = {loss.item(): 5f}, Accuracy = {accuracy.item():.2f}')
  print()
  print('Training Complete.')
  print()
